In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# CONFIGURAÇÕES
# ============================================================

BASE_DIR = "./"   # pasta raiz contendo as subpastas
OUTPUT_DIR = "Trajetorias09-09"
Ts = 0.07  # período de amostragem (70 ms), conforme código do Arduino

# Ordem das colunas no arquivo enviado pelo Arduino (send_data):
# x, y, theta, phi_d, phi_e, phi_d_ref, phi_e_ref, e_a_d, e_a_e
COLUNAS = [
    "x", "y", "theta",
    "phi_d", "phi_e",
    "phi_d_ref", "phi_e_ref",
    "e_a_d", "e_a_e"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# FUNÇÃO DE PLOTAGEM
# ============================================================

def plot_trajetoria(df, titulo, salvar_em=None):
    n = len(df)
    t = np.arange(n) * Ts

    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(4, 3, width_ratios=[1, 1.6, 1])

    # -------- Coluna esquerda: posição e orientação --------
    ax_x = fig.add_subplot(gs[0, 0])
    ax_x.plot(t, df["x"], color="tab:blue")
    ax_x.set_ylabel("x (m)")
    ax_x.grid(alpha=0.3)

    ax_y = fig.add_subplot(gs[1, 0])
    ax_y.plot(t, df["y"], color="tab:green")
    ax_y.set_ylabel("y (m)")
    ax_y.grid(alpha=0.3)

    ax_th = fig.add_subplot(gs[2, 0])
    ax_th.plot(t, np.degrees(df["theta"]), color="tab:orange")
    ax_th.set_ylabel(r"$\theta$ (°)")
    ax_th.grid(alpha=0.3)

    # célula (3,0) fica livre; deixamos o eixo x label na última do bloco esquerdo usado
    ax_th.set_xlabel("t (s)")

    # -------- Centro: trajetória y(x) --------
    ax_traj = fig.add_subplot(gs[:, 1])
    ax_traj.plot(df["x"], df["y"], color="tab:blue", linewidth=2, label="Trajetória executada")
    ax_traj.scatter(df["x"].iloc[0], df["y"].iloc[0], color="green",
                     marker="s", s=80, zorder=5, label="início")
    ax_traj.scatter(df["x"].iloc[-1], df["y"].iloc[-1], color="red",
                     marker="x", s=100, zorder=5, label="fim")

    theta_final_deg = np.degrees(df["theta"].iloc[-1])
    ax_traj.set_title("y(x)")
    ax_traj.text(0.02, 0.95, f"θ = {theta_final_deg:.1f}°",
                 transform=ax_traj.transAxes, color="red",
                 fontsize=10, va="top")
    ax_traj.set_xlabel("x (m)")
    ax_traj.set_ylabel("y (m)")
    ax_traj.axis("equal")
    ax_traj.grid(alpha=0.3)
    ax_traj.legend(loc="lower right", fontsize=8)

    # -------- Coluna direita: esforço de controle (PWM) --------
    ax_pwmD = fig.add_subplot(gs[0, 2])
    ax_pwmD.plot(t, df["e_a_d"], color="tab:red")
    ax_pwmD.set_ylabel("PWM D")
    ax_pwmD.grid(alpha=0.3)

    ax_pwmE = fig.add_subplot(gs[1, 2])
    ax_pwmE.plot(t, df["e_a_e"], color="tab:purple")
    ax_pwmE.set_ylabel("PWM E")
    ax_pwmE.grid(alpha=0.3)

    # -------- Velocidades das rodas (medida vs referência) --------
    ax_wD = fig.add_subplot(gs[2, 2])
    ax_wD.plot(t, df["phi_d"], color="tab:brown", label="medida")
    ax_wD.plot(t, df["phi_d_ref"], color="tab:brown", linestyle="--",
               alpha=0.6, label="setpoint")
    ax_wD.set_ylabel(r"$\varphi_D$ (rad/s)")
    ax_wD.legend(fontsize=7)
    ax_wD.grid(alpha=0.3)

    ax_wE = fig.add_subplot(gs[3, 2])
    ax_wE.plot(t, df["phi_e"], color="tab:cyan", label="medida")
    ax_wE.plot(t, df["phi_e_ref"], color="tab:cyan", linestyle="--",
               alpha=0.6, label="setpoint")
    ax_wE.set_ylabel(r"$\varphi_E$ (rad/s)")
    ax_wE.set_xlabel("t (s)")
    ax_wE.legend(fontsize=7)
    ax_wE.grid(alpha=0.3)

    fig.suptitle(titulo, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if salvar_em:
        fig.savefig(salvar_em, dpi=150)
        plt.close(fig)
    else:
        plt.show()


# ============================================================
# LOOP PRINCIPAL: percorre subpastas e arquivos
# ============================================================

def processar_pasta(base_dir):
    base_path = Path(base_dir)

    # percorre cada subpasta (ex: "chao", "suspenso")
    for subpasta in sorted(base_path.iterdir()):
        if not subpasta.is_dir():
            continue

        for arquivo in sorted(subpasta.glob("*.csv")):
            try:
                df = pd.read_csv(arquivo, header=None, names=COLUNAS)
            except Exception as e:
                print(f"Erro ao ler {arquivo}: {e}")
                continue

            titulo = f"{subpasta.name}-{arquivo.stem}"
            saida = Path(OUTPUT_DIR) / f"{titulo}.png"

            print(f"Processando: {titulo} ({len(df)} amostras)")
            plot_trajetoria(df, titulo, salvar_em=saida)


if __name__ == "__main__":
    processar_pasta(BASE_DIR)
    print(f"\nFiguras salvas em: {OUTPUT_DIR}/")

Processando: Chao-Circ (4729 amostras)
Processando: Chao-Inf (1314 amostras)
Processando: Chao-LSG-1 (1063 amostras)
Processando: Chao-LSG-2 (1120 amostras)
Processando: Chao-semiCirc (981 amostras)
Processando: Chao-sin (5450 amostras)
Processando: Chao-SuperZZ1 (2222 amostras)
Processando: Chao-SuperZZ2 (3067 amostras)
Processando: Chao-ZZx1-inv (1028 amostras)
Processando: Chao-ZZx1 (1037 amostras)
Processando: Chao-ZZx2-inv (614 amostras)
Processando: Chao-zzx2-inv2 (1761 amostras)
Processando: Chao-ZZx2 (1395 amostras)
Processando: Chao-ZZxReto (1058 amostras)
Processando: Chao-ZZy1 (1608 amostras)
Processando: Chao-ZZy2 (1637 amostras)

Figuras salvas em: Trajetorias09-09/
